In [1]:
# PHASE 2 - ABLATION 9b: EXPLAINABLE AI - attention rollout (ViT models) and Grad-CAM (ResNet18, Swin-T) with centre-mass ratio (Part-I protocol) for the frozen kaiko, lunit, phikon features
# For every model: a tumour probe (StandardScaler + LogisticRegression C=1, balanced) is fitted on 30000 stratified patches of centres 0,1,3,4; 200 correctly classified TUMOUR patches are drawn
# from the train centres ("in-domain") and 200 from the held-out centre 2; attention rollout (ViTs: CLS -> patch tokens, heads averaged, residual-corrected, 14x14 map) or Grad-CAM w.r.t. the folded probe logit (ResNet18: last conv, 3x3 map at 96 px; Swin-T: last stage, 7x7 map) and the
# centre-mass ratio = attribution mass in the central 1/3 x 1/3 region that defines the Camelyon17 label (the chance level for each grid is printed).
# Models: ORIGINAL | SimCLR control | self | ADJACENCY | CCSM ms / ms+Fourier (ResNet18 also ms-random) (+ Lunit combined), seeds 42/43/44.  Kaggle: GPU on, Internet on.
# Checkpoints are found/downloaded automatically; for the combined Lunit arm attach the OUTPUT of phase2_ABL5_lunit_combined_adjacency_ccsm (skipped if absent).
# Estimated runtime ~6-7.5 h (51 models).  Resumable; stops before 11 h.  Figures (same 4 fixed patches for every model) are written at the end.
!pip install -q timm datasets kagglehub transformers matplotlib


import os, gc, json, glob, random, time, shutil, warnings, pickle
from collections import defaultdict
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore')
T_START = time.time()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CUR_SIZE = [224]          # network input size of the model currently evaluated (ResNet18 = 96, the others 224)

def load_camelyon():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset("wltjr1007/Camelyon17-WILDS"); parts = [ds[k] for k in ds.keys()]
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"✓ Camelyon17-WILDS: {len(full):,} patches"); return full

def patient_split_by_centre(meta, val_frac, seed, centres=(0, 1, 2, 3, 4)):
    """Per-centre, patient-level split (identical convention to the Part-I probe scripts)."""
    tr, te = {}, {}
    for c in centres:
        pats = sorted(meta.loc[meta['center'] == c, 'patient'].unique().tolist())
        random.Random(seed + c).shuffle(pats)
        if len(pats) < 2: tr[c], te[c] = set(pats), set(); continue
        n_te = min(max(1, int(round(val_frac * len(pats)))), len(pats) - 1)
        te[c], tr[c] = set(pats[:n_te]), set(pats[n_te:])
    return tr, te

def stratified_sample(labels, n_per_class, seed):
    rng = np.random.RandomState(seed)
    pos, neg = np.where(labels == 1)[0], np.where(labels == 0)[0]
    rng.shuffle(pos); rng.shuffle(neg)
    k = min(n_per_class, len(pos), len(neg))
    if k < n_per_class: print(f"   ⚠ only {len(pos)} pos / {len(neg)} neg available -> {k}/class")
    idx = np.concatenate([pos[:k], neg[:k]]); rng.shuffle(idx); return idx

@torch.no_grad()
def embed(enc, X_u8, bs=128, amp=False, size=None):
    """uint8 [N,3,h,w] -> pooled features [N,D] (bilinear up-sampling to `size` on the GPU, ImageNet normalisation)."""
    out = []; size = size or CUR_SIZE[0]
    for i in range(0, len(X_u8), bs):
        x = X_u8[i:i + bs].to(DEVICE).float() / 255
        if x.shape[-1] != size: x = F.interpolate(x, size=(size, size), mode='bilinear', align_corners=False, antialias=True)
        x = (x - MEAN.to(DEVICE)) / STD.to(DEVICE)
        with torch.autocast('cuda', enabled=amp and DEVICE.type == 'cuda'):
            out.append(enc.encode(x).float().cpu())
    return torch.cat(out).numpy()

def load_uint8(ds, indices, size, desc):
    """Decode `indices` of an HF dataset to a uint8 tensor [n,3,size,size] (PIL bilinear resize if needed)."""
    from tqdm.auto import tqdm
    from torchvision import transforms
    tf = transforms.Resize((size, size))
    out = np.zeros((len(indices), size, size, 3), np.uint8)
    for k, i in enumerate(tqdm(indices, desc=desc)):
        im = ds[int(i)]['image']
        if im.mode != 'RGB': im = im.convert('RGB')
        if im.size != (size, size): im = tf(im)
        out[k] = np.asarray(im)
    return torch.from_numpy(out).permute(0, 3, 1, 2).contiguous()


# ---- frozen encoders / checkpoint loading (same code as the D7/D8 evaluations) ----
import os, gc, json, glob, random, time, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm import tqdm

warnings.filterwarnings('ignore')


class EC:
    RUN_FAMILIES = ['kaiko', 'lunit', 'phikon']
    SEEDS = [42, 43, 44]
    BUDGET_H = 11.0                       # stop starting new chunks when the next one would not fit
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CKPT_SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working/checkpoints"]
    KAGGLE_CKPT_DATASETS = ["sethu2954/adjc-kaiko-checkpoints", "sethu2954/adjc-lunit-checkpoints", "sethu2954/adjc-phikon-checkpoints"]   # fallback if not attached
    PRIOR_RESULT_ROOTS = ["/kaggle/input"]                                  # earlier runs' outputs (p2_eval_all/*.json) are reused
    FEATURE_DIR = "/kaggle/working/unused_features"
    RESULTS_DIR = "/kaggle/working/unused_results"
    KEEP_FEATURES = False

    EXTRACT_BATCH = 128
    NUM_WORKERS = 4
    ALL_CENTERS = [0, 1, 2, 3, 4]
    TEST_CENTER = 2
    LR_C, LR_MAX_ITER, PROBE_SEED = 1.0, 2000, 42
    BOOT_N, BOOT_SEED = 1000, 7
    LOCO_FOLDS = [0, 1, 2, 3, 4]
    CID_TRAIN_PER_CENTER, CID_TEST_PER_CENTER, CID_VAL_FRAC = 3000, 1000, 0.25

    FAMILIES = {   # kind, hub id / timm name, input size, fp16 extraction, Part-I LOCO of the original (pipeline check; None = ImageNet control)
        'resnet18':  dict(kind='resnet', hub=None, size=96,  amp=False, ref=0.9528, label='ImageNet ResNet18'),
        'vit_small': dict(kind='timm', hub='vit_small_patch16_224', size=224, amp=False, ref=None, label='ImageNet ViT-S/16'),
        'swin_tiny': dict(kind='timm', hub='swin_tiny_patch4_window7_224', size=224, amp=False, ref=None, label='ImageNet Swin-T'),
        'kaiko':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms', size=224, amp=False, ref=0.9873, label='Kaiko ViT-S/16'),
        'lunit':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.lunit_dino', size=224, amp=False, ref=0.9814, label='Lunit ViT-S/16'),
        'phikon':    dict(kind='hf', hub='owkin/phikon', size=224, amp=True, ref=0.9818, label='Phikon ViT-B/16'),
    }
    ARMS = {}
    DEFAULT_ARMS = ['neighbor', 'self']
    BACKBONE = {'kaiko': 'vit_small', 'lunit': 'vit_small', 'phikon': 'phikon'}      # name used in the checkpoint file names
    PART1_OTHERS = "Part-I LOCO references: Stain Aug 0.9884 | Kaiko 0.9873 | Phikon 0.9818 | Lunit 0.9814 | ImageNet ResNet18 0.9528"

    @staticmethod
    def arms(fam):
        return EC.ARMS.get(fam, EC.DEFAULT_ARMS)

    @staticmethod
    def chunks(fam):
        """[(chunk_models)] : chunk 1 = original + seed 42, then one chunk per further seed.  Model name = '<family>__ctrl' / '<family>__<arm>_s<seed>'."""
        out = []
        for k, s in enumerate(EC.SEEDS):
            c = [f'{fam}__ctrl'] if k == 0 else []
            c += [f'{fam}__{a}_s{s}' for a in EC.arms(fam)]
            out.append(c)
        return out


def parse(name):
    fam, rest = name.split('__')
    if rest == 'ctrl':
        return fam, 'ctrl', None
    arm, s = rest.rsplit('_s', 1)
    return fam, arm, int(s)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ---------------------------------------------------------------- encoders (same construction as Part I / the training notebooks)
class Encoder(nn.Module):
    def __init__(self, fam):
        super().__init__()
        spec = EC.FAMILIES[fam]; self.pool = None
        if spec['kind'] == 'resnet':
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.backbone = nn.Sequential(*list(net.children())[:-1]); self.pool = nn.AdaptiveAvgPool2d(1); self.dim = 512
        elif spec['kind'] == 'hf':
            from transformers import AutoModel
            class PhikonBackbone(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.vit = AutoModel.from_pretrained(spec['hub'])
                def forward(self, x):
                    return self.vit(pixel_values=x).last_hidden_state[:, 0]           # CLS token, as in Part I
            self.backbone = PhikonBackbone(); self.dim = self.backbone.vit.config.hidden_size
        else:
            import timm
            self.backbone = timm.create_model(spec['hub'], pretrained=True, num_classes=0)
            self.dim = self.backbone.num_features

    @torch.no_grad()
    def encode(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f


def ensure_checkpoint_roots():
    if glob.glob("/kaggle/input/**/adjclv4-*_best.pt", recursive=True):
        return
    import kagglehub
    for slug in EC.KAGGLE_CKPT_DATASETS:
        try:
            path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", path)
            EC.CKPT_SEARCH_ROOTS.append(path)
        except Exception as e:
            print(f"⚠ could not download {slug}: {e}")


def find_checkpoint(fam, arm, seed):
    fname = f"adjclv4-{fam}_{EC.BACKBONE[fam]}_{arm}_s{seed}_best.pt"
    for root in EC.CKPT_SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits:
            return hits[0]
    return None


def build_model(name):
    fam, arm, seed = parse(name)
    if arm == 'ctrl':
        return Encoder(fam)
    path = find_checkpoint(fam, arm, seed)
    if path is None:
        return None
    enc = Encoder(fam)
    ck = torch.load(path, map_location='cpu')
    missing, unexpected = enc.load_state_dict(ck['model_state_dict'], strict=False)
    bad = [k for k in unexpected if not k.startswith(('head.', 'adv_head.'))]
    assert not missing and not bad, f"{name}: missing={missing[:3]} unexpected={bad[:3]}"
    print(f"  [{name}] loaded {os.path.basename(path)}  (epoch {ck.get('epoch','?')}, final loss {ck.get('best_val', float('nan')):.4f})")
    return enc


# ---------------------------------------------------------------- one-pass feature extraction
class MultiSizeDataset(Dataset):
    def __init__(self, hf, sizes):
        self.hf, self.sizes = hf, sorted(sizes)
        norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        self.tf = {s: transforms.Compose([transforms.Resize((s, s)), transforms.ToTensor(), norm]) for s in self.sizes}

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, i):
        img = self.hf[int(i)]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {s: self.tf[s](img) for s in self.sizes}


def feat_path(name):
    return f"{EC.FEATURE_DIR}/{name}.npy"


def extract_missing(hf, names, device):
    todo = [n for n in names if not os.path.exists(feat_path(n))]
    if not todo:
        return
    mods = {}
    for n in todo:
        m = build_model(n)
        if m is None:
            print(f"  [{n}] checkpoint not found -> skipped")
            continue
        mods[n] = m.to(device).eval()
    if not mods:
        return
    fam = parse(todo[0])[0]; size = EC.FAMILIES[fam]['size']; amp = EC.FAMILIES[fam]['amp'] and device.type == 'cuda'
    N = len(hf)
    out = {n: np.empty((N, mods[n].dim), dtype=np.float32) for n in mods}
    loader = DataLoader(MultiSizeDataset(hf, {size}), batch_size=EC.EXTRACT_BATCH, shuffle=False,
                        num_workers=EC.NUM_WORKERS, pin_memory=device.type == 'cuda')
    pos, t0 = 0, time.time()
    for batch in tqdm(loader, desc=f"extract {fam}: {len(mods)} models"):
        x = batch[size].to(device, non_blocking=True); B = x.size(0)
        for n, m in mods.items():
            with torch.autocast('cuda', enabled=amp):
                f = m.encode(x)
            out[n][pos:pos + B] = f.float().cpu().numpy()
        pos += B
    assert pos == N, (pos, N)
    os.makedirs(EC.FEATURE_DIR, exist_ok=True)
    for n in mods:
        np.save(feat_path(n), out[n])
    print(f"✓ extracted {list(mods)} in {(time.time()-t0)/60:.1f} min")
    del mods, out; gc.collect(); torch.cuda.empty_cache()


# ---------------------------------------------------------------- evaluation (same recipe as Part I)
def bootstrap_auroc_by_patient(y, p, pat, n_boot, seed):
    pats = np.unique(pat)
    idx_by = {q: np.where(pat == q)[0] for q in pats}
    rng, vals = np.random.RandomState(seed), []
    for _ in range(n_boot):
        s = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by[q] for q in s])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def probe_auroc(X, y, centers, patients, train_c, test_c, with_ci=False):
    tr, te = np.isin(centers, train_c), centers == test_c
    sc = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=EC.LR_MAX_ITER, C=EC.LR_C, class_weight='balanced',
                             random_state=EC.PROBE_SEED, n_jobs=-1)
    clf.fit(sc.transform(X[tr]), y[tr])
    prob = clf.predict_proba(sc.transform(X[te]))[:, 1]
    auc = roc_auc_score(y[te], prob)
    bal = balanced_accuracy_score(y[te], prob > 0.5)
    if with_ci:
        lo, hi = bootstrap_auroc_by_patient(y[te], prob, patients[te], EC.BOOT_N, EC.BOOT_SEED)
        return auc, bal, lo, hi
    return auc, bal


def center_id_probe(X, centers, patients):
    """Patch-level 5-way center-identity probe, patient-disjoint train/test within each center."""
    rng = np.random.RandomState(EC.PROBE_SEED)
    tr_idx, te_idx = [], []
    for c in EC.ALL_CENTERS:
        pats = sorted(np.unique(patients[centers == c]).tolist())
        random.Random(EC.PROBE_SEED + c).shuffle(pats)
        n_te = min(max(1, int(round(EC.CID_VAL_FRAC * len(pats)))), len(pats) - 1)
        te_p, tr_p = set(pats[:n_te]), set(pats[n_te:])
        for pool, n_want, store in ((np.where((centers == c) & np.isin(patients, list(tr_p)))[0], EC.CID_TRAIN_PER_CENTER, tr_idx),
                                    (np.where((centers == c) & np.isin(patients, list(te_p)))[0], EC.CID_TEST_PER_CENTER, te_idx)):
            store.append(rng.choice(pool, size=min(n_want, len(pool)), replace=False))
    tr_idx, te_idx = np.concatenate(tr_idx), np.concatenate(te_idx)
    sc = StandardScaler().fit(X[tr_idx])
    clf = LogisticRegression(max_iter=EC.LR_MAX_ITER, n_jobs=-1).fit(sc.transform(X[tr_idx]), centers[tr_idx])
    return 100 * accuracy_score(centers[te_idx], clf.predict(sc.transform(X[te_idx])))




# ====================================================================
# MODELS: original + {SimCLR control, self, adjacency, CCSM ms / fms, combined}  x seeds  (Kaiko / Lunit / Phikon)
# ====================================================================
FAM_ARMS = {'resnet18':  ['control', 'self', 'neighbor', 'ms', 'msrand', 'fms'],
            'vit_small': ['control', 'self', 'neighbor', 'ms', 'fms'],
            'swin_tiny': ['control', 'self', 'neighbor', 'ms', 'fms'],
            'kaiko':  ['control', 'self', 'neighbor', 'ms', 'fms'],
            'lunit':  ['control', 'self', 'neighbor', 'ms', 'fms', 'comb'],
            'phikon': ['control', 'self', 'neighbor', 'ms', 'fms']}
ARM_LABEL = {'ctrl': 'ORIGINAL pretrained weights', 'msrand': 'CCSM: MixStyle, random partners', 'control': 'SimCLR control (CCSM recipe)', 'self': 'self control (adjacency recipe)', 'neighbor': 'ADJACENCY (Method 1)',
             'ms': 'CCSM: MixStyle', 'fms': 'CCSM: MixStyle + Fourier', 'comb': 'COMBINED adjacency + CCSM'}
ARM_ORDER = ['ctrl', 'control', 'self', 'neighbor', 'ms', 'msrand', 'fms', 'comb']
ADJ_SLUGS = {'resnet18': "peesarisathvikreddy/adjclv2-checkpoints", 'vit_small': "peesarisathvikreddy/adjclv2-checkpoints", 'swin_tiny': "peesarisathvikreddy/adjclv2-checkpoints",
             'kaiko': "sethu2954/adjc-kaiko-checkpoints", 'lunit': "sethu2954/adjc-lunit-checkpoints", 'phikon': "sethu2954/adjc-phikon-checkpoints"}
COMB_SLUG = "badrinathachanta/ablation-combined-checkpoints"      # Lunit combined (adjacency + CCSM) checkpoints from ABL5
CCSM_SLUGS = {'resnet18': "badrinathachanta/ccsm-checkpoints-1", 'vit_small': "badrinathachanta/ccsm-checkpoints-1", 'swin_tiny': "badrinathachanta/ccsm-checkpoints-1",
              'kaiko': "badrinathachanta/ccsm-checkpoints-2", 'lunit': "badrinathachanta/ccsm-checkpoints-2", 'phikon': "badrinathachanta/ccsm-checkpoints-2"}

def ckpt_fname(fam, arm, seed):
    if arm in ('self', 'neighbor') and fam in ('resnet18', 'vit_small', 'swin_tiny'): return f"adjclv2_{fam}_{arm}_s{seed}_best.pt"      # adjacency v2 recipe (ResNet18 / ViT-S / Swin-T)
    if arm in ('self', 'neighbor'): return f"adjclv4-{fam}_{'phikon' if fam == 'phikon' else 'vit_small'}_{arm}_s{seed}_best.pt"
    if arm in ('control', 'ms', 'msrand', 'fms'): return f"ccsm_{fam}_{arm}_s{seed}_best.pt"
    if arm == 'comb': return f"abl5-comb_vit_small_neighbor_s{seed}_best.pt"
    raise ValueError(arm)

def ensure_checkpoint_roots():
    import kagglehub
    for fam in RUN_FAMILIES:
        pairs = [('ccsm', CCSM_SLUGS[fam])] + ([('adjacency', ADJ_SLUGS[fam])] if fam in ADJ_SLUGS else []) + ([('comb', COMB_SLUG)] if fam == 'lunit' else [])
        for kind, slug in pairs:
            probe = ckpt_fname(fam, {'adjacency': 'neighbor', 'comb': 'comb'}.get(kind, 'ms'), SEEDS[0])
            if glob.glob(os.path.join('/kaggle/input', '**', probe), recursive=True): continue
            try:
                path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", slug); EC.CKPT_SEARCH_ROOTS.append(path)
            except Exception as e:
                print(f"⚠ could not download {slug}: {e}")

def find_checkpoint(fam, arm, seed):
    fname = ckpt_fname(fam, arm, seed)
    for root in ['/kaggle/input'] + list(EC.CKPT_SEARCH_ROOTS):
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits: return hits[0]
    return None

def model_names():
    out = []
    for fam in RUN_FAMILIES:
        out.append(f'{fam}__ctrl')
        for s in SEEDS: out += [f'{fam}__{a}_s{s}' for a in FAM_ARMS[fam]]
    return out


def have_result(name): return os.path.exists(f"{RES_DIR}/{name}.json")
def prior_results():
    n = 0
    for p in glob.glob(f'/kaggle/input/**/{os.path.basename(RES_DIR)}/*.json', recursive=True):
        dst = os.path.join(RES_DIR, os.path.basename(p))
        if not os.path.exists(dst): shutil.copy2(p, dst); n += 1
    if n: print(f"✓ reused {n} finished results from an earlier run")

def _ms(v):
    v = np.array(v, dtype=float); return v.mean(), (v.std(ddof=1) if len(v) > 1 else float('nan'))

def run_all(process_fn, device):
    """process_fn(name, encoder) -> dict (JSON-able).  Resumable; stops cleanly before BUDGET_H."""
    T0 = time.time(); last = 0.0
    os.makedirs(RES_DIR, exist_ok=True); prior_results(); ensure_checkpoint_roots()
    names = model_names(); todo = [n for n in names if not have_result(n)]
    print(f"{len(names) - len(todo)}/{len(names)} models already done")
    for n in todo:
        el = (time.time() - T0) / 3600
        if el + max(last, 0.15) > BUDGET_H:
            print(f"\n⏸ time budget reached ({el:.1f} h). Save this version and re-run with this OUTPUT attached as an input to finish the rest."); break
        t0 = time.time()
        enc = build_model(n)
        if enc is None:
            print(f"  [{n}] checkpoint not found -> skipped"); continue
        enc = enc.to(device).eval(); CUR_SIZE[0] = EC.FAMILIES[n.split('__')[0]]['size']
        res = process_fn(n, enc); res['model'] = n
        json.dump(res, open(f"{RES_DIR}/{n}.json", 'w'))
        del enc; gc.collect(); torch.cuda.empty_cache()
        last = (time.time() - t0) / 3600


"""
Deviation from Part-I (documented): Part-I fitted the probe on ALL cached features; here the probe is fitted on 30000 stratified training patches (centres 0,1,3,4) so that no full-dataset feature
extraction is needed per model.  Correct-tumour patches are then drawn from candidate pools (4000 tumour patches per group), seed 123.  ViTs use attention rollout, ResNet18 and Swin-T (no CLS attention) use Grad-CAM; the two kinds of map and their grid sizes differ, so compare arms WITHIN a family, not across families.
"""
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

RUN_FAMILIES = ['kaiko', 'lunit', 'phikon']
SEEDS = [42, 43, 44]
BUDGET_H = 11.0
RES_DIR = "/kaggle/working/abl_xai_b"
MODE = 'mixed'
CACHE = "/kaggle/working/xai_data.pt"
SEED = 42; XAI_SEED = 123; N_PROBE_PER_CLASS = 15000; N_POOL = 4000; N_SAMPLES = 200; CENTER_FRAC = 1.0 / 3.0
TRAIN_CENTERS = [0, 1, 3, 4]; TEST_CENTER = 2
DATA = {}

def prepare():
    for p in [CACHE] + glob.glob('/kaggle/input/**/xai_data.pt', recursive=True):
        if os.path.exists(p):
            print("✓ loading cached data", p); DATA.update(torch.load(p, weights_only=False)); return
    cam = load_camelyon()
    meta = cam.select_columns(['center', 'label']).to_pandas(); cen = meta['center'].to_numpy(); y = meta['label'].to_numpy().astype(np.int64)
    rng = np.random.RandomState(SEED)
    tr_pool = np.where(np.isin(cen, TRAIN_CENTERS))[0]
    probe_idx = tr_pool[stratified_sample(y[tr_pool], N_PROBE_PER_CLASS, SEED)]
    tum_in = np.where(np.isin(cen, TRAIN_CENTERS) & (y == 1))[0]; tum_ho = np.where((cen == TEST_CENTER) & (y == 1))[0]
    in_idx = rng.choice(tum_in, N_POOL, replace=False); ho_idx = rng.choice(tum_ho, min(N_POOL, len(tum_ho)), replace=False)
    all_idx = np.concatenate([probe_idx, in_idx, ho_idx])
    imgs = load_uint8(cam, all_idx, 96, 'patches')
    n1, n2 = len(probe_idx), len(probe_idx) + len(in_idx)
    DATA.update(imgs=imgs, probe_y=y[probe_idx], n1=n1, n2=n2, fixed=[int(n1), int(n1) + 1, int(n2), int(n2) + 1])
    torch.save(dict(DATA), CACHE); print(f"✓ cached {len(all_idx)} patches")

def center_mass_ratio(h):
    H, W = h.shape; ch, cw = max(1, int(round(H * CENTER_FRAC))), max(1, int(round(W * CENTER_FRAC)))
    y0, x0 = (H - ch) // 2, (W - cw) // 2
    return float(h[y0:y0 + ch, x0:x0 + cw].sum() / (h.sum() + 1e-8))

def rollout_from_attn(attn_maps, device):
    n = attn_maps[0].shape[-1]; result = torch.eye(n, device=device).unsqueeze(0)
    for A in attn_maps:
        A = A.mean(dim=1) + torch.eye(n, device=device).unsqueeze(0)
        A = A / A.sum(dim=-1, keepdim=True); result = torch.bmm(A, result)
    side = int(round((n - 1) ** 0.5)); h = result[0, 0, 1:][:side * side].reshape(side, side).float().cpu().numpy()
    return (h - h.min()) / (h.max() - h.min() + 1e-8)

def rollout_timm(vit, x):
    maps = []; restore = []
    for blk in vit.blocks:
        if hasattr(blk.attn, 'fused_attn'): restore.append((blk.attn, blk.attn.fused_attn)); blk.attn.fused_attn = False
    hs = [blk.attn.attn_drop.register_forward_hook(lambda m, i, o: maps.append(i[0].detach())) for blk in vit.blocks]
    try:
        with torch.no_grad(): vit(x)
    finally:
        for h in hs: h.remove()
        for m, o in restore: m.fused_attn = o
    if not maps: raise RuntimeError("no attention captured")
    return rollout_from_attn(maps, x.device)

class PhikonEager(nn.Module):
    """owkin/phikon with eager attention (needed for output_attentions), CLS token as feature; checkpoint keys 'backbone.vit.*' are loaded into .vit."""
    def __init__(self, ckpt=None):
        super().__init__()
        from transformers import AutoModel
        self.vit = AutoModel.from_pretrained('owkin/phikon', attn_implementation='eager')
        if ckpt is not None:
            sd = torch.load(ckpt, map_location='cpu')['model_state_dict']
            sd = {k[len('backbone.vit.'):]: v for k, v in sd.items() if k.startswith('backbone.vit.')}
            missing, unexpected = self.vit.load_state_dict(sd, strict=False); assert not missing, missing[:3]
    @torch.no_grad()
    def encode(self, x): return self.vit(pixel_values=x).last_hidden_state[:, 0]
    def rollout(self, x):
        with torch.no_grad(): out = self.vit(pixel_values=x, output_attentions=True)
        return rollout_from_attn(list(out.attentions), x.device)

_build_model_enc = build_model
def build_model(name):
    fam, arm, seed = parse(name)
    if fam != 'phikon': return _build_model_enc(name)
    if arm == 'ctrl': return PhikonEager(None)
    p = find_checkpoint(fam, arm, seed)
    return PhikonEager(p) if p else None

def fold_probe(clf, sc):
    """StandardScaler folded into the logistic regression, so that logit = eff_w . raw_feature + eff_b (Part-I)."""
    w = clf.coef_[0]; b = float(clf.intercept_[0])
    return torch.tensor(w / sc.scale_, dtype=torch.float32), torch.tensor(b - float(np.sum(w * sc.mean_ / sc.scale_)), dtype=torch.float32)

def grad_cam(enc, x, eff_w, eff_b):
    """Grad-CAM of the tumour-probe logit. ResNet18: last conv block (backbone[7]); Swin-T: last stage (channels-last output)."""
    layer = enc.backbone[7] if isinstance(enc.backbone, nn.Sequential) else enc.backbone.layers[-1]
    box = {}
    def hook(m, i, o): o.retain_grad(); box['a'] = o
    h = layer.register_forward_hook(hook)
    try:
        enc.zero_grad(set_to_none=True)
        with torch.enable_grad():
            f = enc.backbone(x)
            if f.dim() == 4: f = torch.flatten(enc.pool(f), 1) if enc.pool is not None else f.mean((1, 2))
            elif f.dim() == 3: f = f.mean(1)
            logit = f[0] @ eff_w.to(x.device) + eff_b.to(x.device)
            logit.backward()
        act, grad = box['a'][0], box['a'].grad[0]
        if act.dim() == 3 and act.shape[-1] > act.shape[0]: act, grad = act.permute(2, 0, 1), grad.permute(2, 0, 1)
        elif act.dim() == 2:
            n, c = act.shape; side = int(round(n ** 0.5)); act, grad = act.reshape(side, side, c).permute(2, 0, 1), grad.reshape(side, side, c).permute(2, 0, 1)
        cam = F.relu((grad.mean(dim=(1, 2))[:, None, None] * act).sum(0)).detach().float().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    finally:
        h.remove()

def heat(enc, x, eff_w=None, eff_b=None):
    if not isinstance(enc, PhikonEager) and not hasattr(enc.backbone, 'blocks'): return grad_cam(enc, x, eff_w, eff_b)     # ResNet18 / Swin-T
    return enc.rollout(x) if isinstance(enc, PhikonEager) else rollout_timm(enc.backbone, x)

def to_input(u8):
    x = u8.unsqueeze(0).to(DEVICE).float() / 255
    x = F.interpolate(x, size=(CUR_SIZE[0], CUR_SIZE[0]), mode='bilinear', align_corners=False, antialias=True)
    return (x - MEAN.to(DEVICE)) / STD.to(DEVICE)

def process(name, enc):
    fam = name.split('__')[0]; amp = EC.FAMILIES[fam]['amp']; t0 = time.time(); d = DATA; n1, n2 = d['n1'], d['n2']
    Fa = embed(enc, d['imgs'], amp=amp)
    sc = StandardScaler().fit(Fa[:n1]); clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced').fit(sc.transform(Fa[:n1]), d['probe_y'])
    pred = clf.predict(sc.transform(Fa[n1:])); ew, eb = fold_probe(clf, sc)
    pools = {'in_domain': np.arange(n1, n2)[pred[:n2 - n1] == 1], 'held_out': np.arange(n2, len(Fa))[pred[n2 - n1:] == 1]}
    rng = np.random.RandomState(XAI_SEED); res = {'cmr': {}, 'n_candidates': {k: int(len(v)) for k, v in pools.items()}, 'heat_fixed': {}}
    enc.eval()
    for split, pool in pools.items():
        sel = rng.choice(pool, size=min(N_SAMPLES, len(pool)), replace=False) if len(pool) else np.array([], int)
        vals = []
        for i in sel:
            try:
                hm = heat(enc, to_input(d['imgs'][int(i)]), ew, eb); res['grid'] = int(hm.shape[0]); vals.append(center_mass_ratio(hm))
            except Exception as e:
                print("   explain failed:", str(e)[:100]); break
        res['cmr'][split] = vals
    for j in d['fixed']:
        try: res['heat_fixed'][str(j)] = heat(enc, to_input(d['imgs'][j]), ew, eb).tolist()
        except Exception: pass
    m = {k: (np.mean(v) if v else float('nan')) for k, v in res['cmr'].items()}
    print(f"  [{name}] centre-mass ratio: in-domain {m.get('in_domain', float('nan')):.3f} (n={len(res['cmr'].get('in_domain', []))}) | held-out {m.get('held_out', float('nan')):.3f} (n={len(res['cmr'].get('held_out', []))}) | {(time.time()-t0)/60:.1f} min")
    return res

def summarize():
    res = {}
    for p in glob.glob(f"{RES_DIR}/*.json"):
        r = json.load(open(p)); res[r['model']] = r
    print("\n" + "=" * 118 + "\nCENTRE-MASS RATIO of attention rollout (mean over 200 correctly classified tumour patches per split; mean ± std over seeds of the per-seed mean)\n" + "=" * 118)
    rows = []
    for fam in RUN_FAMILIES:
        gs = [r.get('grid') for n, r in res.items() if parse(n)[0] == fam and r.get('grid')]
        ch_ = max(1, int(round(gs[0] / 3))) if gs else 0
        print(f"\n--- {fam} ({gs[0]}x{gs[0]} map, chance of the centre region = {ch_ * ch_ / (gs[0] * gs[0]):.3f}) ---\n  {'':34s}{'in-domain':>16s}{'held-out centre':>18s}{'difference':>14s}" if gs else f"\n--- {fam} ---")
        per = {a: {} for a in ARM_ORDER}
        for n, r in res.items():
            f, a, s = parse(n)
            if f == fam: per[a][s] = r
        for a in ARM_ORDER:
            dd = per[a]
            if not dd: continue
            vals = list(dd.values()); i = [np.mean(v['cmr']['in_domain']) for v in vals if v['cmr'].get('in_domain')]; h = [np.mean(v['cmr']['held_out']) for v in vals if v['cmr'].get('held_out')]
            if not i or not h: continue
            (mi, si), (mh, sh) = _ms(i), _ms(h)
            f_ = lambda m, s: f"{m:.3f}" + (f"±{s:.3f}" if len(vals) > 1 else "")
            print(f"  {ARM_LABEL[a]:34s}{f_(mi, si):>16s}{f_(mh, sh):>18s}{mh - mi:>+14.3f}")
            rows.append(dict(family=fam, method=ARM_LABEL[a], seeds=len(vals), in_domain=f_(mi, si), held_out=f_(mh, sh)))
    pd.DataFrame(rows).to_csv(f"{RES_DIR}/cmr_summary.csv", index=False)
    print("\nA ratio near the chance level means the map is spread uniformly; clearly above it means the explanation sits on the label-defining centre.  Part-I (attention rollout, ViT-S): Kaiko 0.134 / 0.134, Phikon 0.113 / 0.125, Lunit 0.111 / 0.112 (in-domain / held-out).")
    make_figures(res)

def make_figures(res):
    try:
        import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    except Exception: return
    d = DATA
    for fam in RUN_FAMILIES:
        arms = [a for a in ARM_ORDER if (f'{fam}__ctrl' if a == 'ctrl' else f'{fam}__{a}_s42') in res]
        if not arms: continue
        fig, axes = plt.subplots(4, len(arms), figsize=(2.1 * len(arms), 8.6), squeeze=False)
        for r, j in enumerate(d['fixed']):
            img = d['imgs'][j].permute(1, 2, 0).numpy()
            for c, a in enumerate(arms):
                ax = axes[r][c]; ax.axis('off'); ax.imshow(img)
                hm = res[f'{fam}__ctrl' if a == 'ctrl' else f'{fam}__{a}_s42']['heat_fixed'].get(str(j))
                if hm is not None:
                    hm = np.array(hm); hm = F.interpolate(torch.tensor(hm)[None, None], size=img.shape[:2], mode='bilinear', align_corners=False)[0, 0].numpy()
                    ax.imshow(hm, cmap='jet', alpha=0.45)
                if r == 0: ax.set_title(ARM_LABEL[a].split('(')[0].strip()[:18], fontsize=7)
        fig.suptitle(f'{fam}: attention rollout on four fixed tumour patches (rows 1-2 train centres, rows 3-4 held-out centre 2; seed 42)', fontsize=9)
        plt.tight_layout(rect=[0, 0, 1, 0.97]); out = f"{RES_DIR}/rollout_{fam}.png"; plt.savefig(out, dpi=130); plt.close(fig); print("✓", out)

if True:
    os.makedirs(RES_DIR, exist_ok=True)
    prepare()
    run_all(process, DEVICE)
    summarize()


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

  [phikon__neighbor_s44] centre-mass ratio: in-domain 0.125 (n=200) | held-out 0.137 (n=200) | 2.3 min


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  [phikon__ms_s44] centre-mass ratio: in-domain 0.132 (n=200) | held-out 0.128 (n=200) | 2.3 min


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  [phikon__fms_s44] centre-mass ratio: in-domain 0.133 (n=200) | held-out 0.129 (n=200) | 2.3 min

CENTRE-MASS RATIO of attention rollout (mean over 200 correctly classified tumour patches per split; mean ± std over seeds of the per-seed mean)

--- kaiko (14x14 map, chance of the centre region = 0.128) ---
                                           in-domain   held-out centre    difference
  ORIGINAL pretrained weights                  0.117             0.136        +0.019
  SimCLR control (CCSM recipe)           0.144±0.004       0.144±0.006        -0.000
  self control (adjacency recipe)        0.143±0.006       0.151±0.004        +0.009
  ADJACENCY (Method 1)                   0.124±0.003       0.129±0.002        +0.005
  CCSM: MixStyle                         0.137±0.001       0.141±0.007        +0.004
  CCSM: MixStyle + Fourier               0.134±0.005       0.146±0.007        +0.012

--- lunit (14x14 map, chance of the centre region = 0.128) ---
                                 